<a href="https://colab.research.google.com/github/KjelleJ/enkla-ai-experiment/blob/main/AIX_12_nostalgi_quiz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# Nostalgi-Quiz
En enkel prototyp som använder ett API (programmeringsinterface) till den stora språkmodellen gemini-3.5-flash.
---




För att kunna använda api:et behöver du skapa en kostnadsfri API-nyckel till Gemini enligt följande anvisningar. Om du redan har en nyckel till Gemini kan du förstås använda den.

### 🔑 Instruktion för elever: Skapa en egen API-nyckel för Gemini

Följ dessa enkla steg för att skapa din egen kostnadsfria API-nyckel så att du kan köra din pizzeria-chatbot:

1. **Gå till Google AI Studio**
   Öppna webbläsaren och gå till [https://aistudio.google.com/](https://aistudio.google.com/).

2. **Logga in**
   Logga in med ditt vanliga Google-konto (t.ex. ditt skolkonto eller privata Gmail-konto).

3. **Skapa nyckeln (Get API Key)**
   * Klicka på den lilla nyckeln **"Get API key"** nere till vänster.
   * Klicka på **"Create API key"**.
   * Om du får välja projekt, välj **"Create API key in new project"**.

4. **Kopiera nyckeln**
   * En lång rad med bokstäver och siffror visas (din unika API-nyckel).
   * Klicka på **"Copy"** för att kopiera den. *Dela aldrig denna nyckel med någon annan!*

5. **Lägg till nyckeln i Google Colab**
   * Gå tillbaka till denna Google Colab-flik.
   * Klicka på **nyckelsymbolen (🔑 Secrets)** i menyn längst till vänster.
   * Klicka på **"Add new secret"**.
   * Under **Name** skriver du: `GOOGLE_API_KEY`
   * Under **Value** klistrar du in din kopierade API-nyckel.
   * Se till att aktivera bocken för **"Notebook access"** så att koden i din notebook kan läsa nyckeln.

Nu är du redo att köra koden och prata med din chatbot!

In [ ]:
# installera nödvändiga bibliotek
!pip install google-genai

Nu kan vi hämta din API-nyckel från Colabs hemligheter, initiera klienten och starta en chatt med en **systeminstruktion** som beskriver hur chatboten ska bete sig.

In [ ]:
from google import genai
from google.genai import types
from google.colab import userdata

try:
    # Hämtar din API-nyckel från Colabs Secrets manager
    api_key = userdata.get('GOOGLE_API_KEY')
    client = genai.Client(api_key=api_key)
    print("Klienten har initierats framgångsrikt!")
except Exception as e:
    print("Fel: Kontrollera att du har lagt till GOOGLE_API_KEY i dina Secrets i Colab.")
    print(e)

---
# Sätt temperatur och skapa systemprompt
Pröva först botten med ursprungliga definitioner och sedan med definitioner som ger en mer korrekt och pålitlig bot.

In [ ]:
# Ursprunglig temperatur och systemprompt
import textwrap
from google import genai
from google.genai import types
from google.colab import userdata

# Sätt temperatur - hur ska boten vara?
# Låg temperatur (0.0 – 0.3) – Fokuserad och konsekvent
# Medelhög temperatur (0.4 – 0.8) – Balanserad och naturlig (Rekommenderas!)
# Hög temperatur (0.9 – 1.5) – Kreativ och oväntad
temperature = 0.7

# Definiera systeminstruktionen
system_instruction_quiz = """
Du är 'Minnes-Kompisen', en varm, tålmodig och uppmuntrande spelledare för ett
nostalgiquiz om Sverige under 1950-, 1960- och 1970-talen.

Regler för spelet:
1. Du ska ställa EN fråga i taget. Frågorna ska handla om musik, film, historiska
händelser, vardagsliv eller kända profiler från dessa årtionden.
2. Vänta på spelarens svar innan du går vidare.
3. Om spelaren svarar fel eller inte minns, ge en vänlig ledtråd istället för att
bara ge svaret direkt.
4. När spelaren svarar rätt, ge beröm och berätta kortfattat en rolig eller
intressant kuriosa (max 3 meningar) relaterad till ämnet för att väcka nostalgi.
Ställ sedan nästa fråga.
5. Håll en varm, artig och nostalgisk ton. Använd ett enkelt och tydligt svenskt språk.

Börja med att hälsa spelaren varmt välkommen, förklara kort vad vi ska göra och
ställ den allra första frågan!
"""

In [ ]:
# Ändrad systemprompt och temperatur.
# Ger en mera korrekt och pålitlig spelledare.
import textwrap
from google import genai
from google.genai import types
from google.colab import userdata

# Sätt temperatur - hur ska boten vara?
# Låg temperatur (0.0 – 0.3) – Fokuserad och konsekvent
# Medelhög temperatur (0.4 – 0.8) – Balanserad och naturlig (Rekommenderas!)
# Hög temperatur (0.9 – 1.5) – Kreativ och oväntad
temperature = 0.3

# 1. Definiera systeminstruktionen

# Definiera systeminstruktionen
system_instruction_quiz = """
Du är en tålmodig och uppmuntrande spelledare för ett
nostalgiquiz om Sverige under 1950-, 1960- och 1970-talen.

Regler för spelet:
1. Du ska ställa EN fråga i taget. Frågorna ska handla om musik, film, historiska
händelser, vardagsliv eller kända profiler från dessa årtionden.
2. Vänta på spelarens svar innan du går vidare.
3. Om spelaren svarar fel eller inte minns, ge en vänlig ledtråd istället för att
bara ge svaret direkt.
4. När spelaren svarar rätt, ge beröm och berätta kortfattat en rolig eller
intressant kuriosa (max 3 meningar) relaterad till ämnet.
Ställ sedan nästa fråga.
5. Håll en artig och respektfull ton. INGA skämt. Använd ett enkelt och tydligt svenskt språk.

Börja med att hälsa spelaren välkommen, förklara kort vad vi ska göra och
ställ den allra första frågan!
"""

---
#Definiera en funktion för vårt quiz
Ibland inget svar från Gemini och sessionen bryts. Enligt Gemini kan det bero på att ett säkerhetsfilter slår till (upphovsrätt, farliga ord,...). Funktionen quiz() försöker fixa det genom att öppna upp filtren. Om det inte räcker sänds ett för oss osynligt meddelande som ber Gemini att omformulera svaret. Om det inte heller hjälper får vi ett meddelande "Oj, mitt minne svek mig lite där! Låt oss gå vidare till nästa fråga...". Sessionen bryts inte.

In [ ]:
def quiz():
  try:
      # 2. Starta klienten och chatten med generösa säkerhetsinställningar
      api_key = userdata.get('GOOGLE_API_KEY')
      client = genai.Client(api_key=api_key)

      friendly_safety = [
          types.SafetySetting(
              category=types.HarmCategory.HARM_CATEGORY_HARASSMENT,
              threshold=types.HarmBlockThreshold.BLOCK_NONE,
          ),
          types.SafetySetting(
              category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
              threshold=types.HarmBlockThreshold.BLOCK_NONE,
          ),
          types.SafetySetting(
              category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
              threshold=types.HarmBlockThreshold.BLOCK_NONE,
          ),
          types.SafetySetting(
              category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
              threshold=types.HarmBlockThreshold.BLOCK_NONE,
          ),
      ]

      quiz_chat = client.chats.create(
          model="gemini-3.5-flash-lite",
          config=types.GenerateContentConfig(
              system_instruction=system_instruction_quiz,
              temperature=temperature,
              safety_settings=friendly_safety
          )
      )

      # Hämta och visa välkomsthälsningen med den första frågan
      welcome_response = quiz_chat.send_message("Hej! Jag är redo att börja quizzet.")

      if welcome_response.text:
          formatted_welcome = "\n".join(
              textwrap.fill(line, width=80) if line.strip() else ""
              for line in welcome_response.text.split("\n")
          )
          print("Spelledaren:\n" + formatted_welcome + "\n")

      # Starta den interaktiva loopen direkt efteråt
      print("Skriv 'avsluta' för att avbryta quizzet.\n")

      while True:
          user_answer = input("Ditt svar: ")
          if user_answer.lower() == 'avsluta':
              print("\nSpelledaren: Tack för idag! Det var fantastiskt roligt att minnas tillbaka tillsammans med dig. Välkommen åter!")
              break

          # Skicka svaret till chatten
          response = quiz_chat.send_message(user_answer)

          # SÄKERHETSÅTGÄRD: Om svaret blev blockerat och är tomt, gör vi ett nytt försök med en tvingande instruktion
          if not response.text:
              # Vi skickar en osynlig signal till chatten att svara enklare utan skyddade citat/namn
              # Osynlig prompt / meta-instruktion
              response = quiz_chat.send_message(
                  "(Ditt förra svar blockerades av ett filter. Svara på min förra input igen, men skriv ett väldigt kort svar, sammanfatta fritt med egna ord och undvik exakta citat eller skyddade sångtexter.)"
              )

          # Skriv ut svaret om det lyckades nu
          if response.text:
              formatted_response = "\n".join(
                  textwrap.fill(line, width=80) if line.strip() else ""
                  for line in response.text.split("\n")
              )
              print(f"\nSpelledaren:\n{formatted_response}\n")
          else:
              print("\nSpelledaren: Oj, mitt minne svek mig lite där! Låt oss gå vidare till nästa fråga. Skriv vad som helst för att fortsätta!\n")

  except Exception as e:
      print("Ett fel uppstod. Kontrollera att du har lagt till GOOGLE_API_KEY i dina Secrets:", e)

---
#Starta quiz

In [ ]:
quiz()